<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [4]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed):
    X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

Não é estritamente necessário normalizar os dados para esse tipo de modelo baseado em árvores, como Random Forest e AdaBoost com árvores fracas, porque esses modelos não dependem da escala das variáveis da mesma forma que modelos lineares, SVMs ou KNN. Eles fazem divisões com base em limiares dos atributos, e não em distâncias ou magnitude absoluta.

Ainda assim, manter um pipeline consistente é importante. Para modelos de árvore, a normalização geralmente não traz ganho relevante de desempenho.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [1]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

def train_random_forest(X_train, y_train, seed):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed):
    base_tree = DecisionTreeClassifier(max_depth=1, random_state=seed)

    try:
        model = AdaBoostClassifier(
            estimator=base_tree,
            n_estimators=50,
            random_state=seed
        )
    except TypeError:
        model = AdaBoostClassifier(
            base_estimator=base_tree,
            n_estimators=50,
            random_state=seed
        )

    model.fit(X_train, y_train)
    return model

As funções foram implementadas com modelos do sklearn e uso de random_state para garantir reprodutibilidade. O Random Forest utiliza várias árvores, enquanto o AdaBoost usa classificadores fracos em sequência.

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [3]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

A função evaluate realiza as predições no conjunto de teste e retorna a acurácia do modelo.

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [5]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")

    acc = evaluate(model, X_test, y_test)
    return acc

In [6]:
print("RF:", run_pipeline("rf", 42))
print("AdaBoost:", run_pipeline("ab", 42))

RF: 0.9661428571428572
AdaBoost: 0.6681428571428571


**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

def full_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }

X_train, X_test, y_train, y_test = load_data(42)

rf_model = train_random_forest(X_train, y_train, 42)
ab_model = train_adaboost(X_train, y_train, 42)

rf_metrics = full_metrics(rf_model, X_test, y_test)
ab_metrics = full_metrics(ab_model, X_test, y_test)

results_q5 = pd.DataFrame([
    {"model": "Random Forest", **rf_metrics},
    {"model": "AdaBoost", **ab_metrics},
])

results_q5

,model,accuracy,precision,recall,f1
0,Random Forest,0.966143,0.966138,0.966143,0.966116
1,AdaBoost,0.668143,0.688355,0.668143,0.671135


O modelo com melhor desempenho inicial deve ser identificado observando a tabela de métricas. Em geral, o Random Forest tende a apresentar desempenho mais forte e estável nesse tipo de tarefa, mas a conclusão deve ser baseada nos valores obtidos no experimento.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [8]:
seeds = [42, 7]
results_q6 = []

for seed in seeds:
    X_train, X_test, y_train, y_test = load_data(seed)

    rf_model = train_random_forest(X_train, y_train, seed)
    ab_model = train_adaboost(X_train, y_train, seed)

    rf_metrics = full_metrics(rf_model, X_test, y_test)
    ab_metrics = full_metrics(ab_model, X_test, y_test)

    results_q6.append({"seed": seed, "model": "Random Forest", **rf_metrics})
    results_q6.append({"seed": seed, "model": "AdaBoost", **ab_metrics})

pd.DataFrame(results_q6)

,seed,model,accuracy,precision,recall,f1
0,42,Random Forest,0.966143,0.966138,0.966143,0.966116
1,42,AdaBoost,0.668143,0.688355,0.668143,0.671135
2,7,Random Forest,0.968214,0.968212,0.968214,0.968190
3,7,AdaBoost,0.611500,0.665315,0.611500,0.616042


Os resultados podem mudar levemente ao alterar a seed, porque a separação treino/teste e o treinamento dos modelos dependem de aleatoriedade controlada. Ainda assim, o experimento é reproduzível porque, para uma mesma seed, os resultados devem ser sempre os mesmos.

Portanto, reprodutibilidade não significa que seeds diferentes geram resultados idênticos, mas sim que o mesmo experimento pode ser repetido sob as mesmas condições e produzir o mesmo resultado.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [9]:
def train_test_accuracy(model, X_train, y_train, X_test, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    return {
        "train_accuracy": accuracy_score(y_train, y_pred_train),
        "test_accuracy": accuracy_score(y_test, y_pred_test)
    }

X_train, X_test, y_train, y_test = load_data(42)

rf_model = train_random_forest(X_train, y_train, 42)
ab_model = train_adaboost(X_train, y_train, 42)

overfit_results = pd.DataFrame([
    {"model": "Random Forest", **train_test_accuracy(rf_model, X_train, y_train, X_test, y_test)},
    {"model": "AdaBoost", **train_test_accuracy(ab_model, X_train, y_train, X_test, y_test)},
])

overfit_results["gap"] = overfit_results["train_accuracy"] - overfit_results["test_accuracy"]
overfit_results

,model,train_accuracy,test_accuracy,gap
0,Random Forest,1.000000,0.966143,0.033857
1,AdaBoost,0.672964,0.668143,0.004821


Existe indício de overfitting quando a acurácia no treino é significativamente maior do que a acurácia no teste. Isso mostra que o modelo aprendeu muito bem os dados vistos, mas generaliza pior para dados novos.

Entre os dois modelos, o Random Forest pode atingir acurácia de treino muito alta, especialmente com árvores profundas, o que pode aumentar esse gap. O AdaBoost também pode sofrer com isso, mas o comportamento deve ser avaliado pelos resultados obtidos.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
results_q8 = []

X_train, X_test, y_train, y_test = load_data(42)

for n in [10, 50, 100]:
    rf_model = RandomForestClassifier(
        n_estimators=n,
        random_state=42,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)

    base_tree = DecisionTreeClassifier(max_depth=1, random_state=42)
    try:
        ab_model = AdaBoostClassifier(
            estimator=base_tree,
            n_estimators=n,
            random_state=42
        )
    except TypeError:
        ab_model = AdaBoostClassifier(
            base_estimator=base_tree,
            n_estimators=n,
            random_state=42
        )

    ab_model.fit(X_train, y_train)

    rf_acc = accuracy_score(y_test, rf_model.predict(X_test))
    ab_acc = accuracy_score(y_test, ab_model.predict(X_test))

    results_q8.append({"model": "Random Forest", "n_estimators": n, "accuracy": rf_acc})
    results_q8.append({"model": "AdaBoost", "n_estimators": n, "accuracy": ab_acc})

pd.DataFrame(results_q8)

A variação de n_estimators permite observar se adicionar mais estimadores melhora o desempenho ou apenas aumenta custo computacional. Se a acurácia variar pouco, isso indica baixa sensibilidade ao hiperparâmetro nessa faixa.

O modelo mais sensível será aquele cuja acurácia oscilar mais conforme n_estimators muda. Isso deve ser concluído com base na tabela gerada.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. A acurácia não é suficiente sozinha para avaliar os modelos. Ela resume a proporção de acertos, mas pode esconder falhas em classes específicas. Por isso, é importante observar também precisão, recall e F1-score, que oferecem uma visão mais completa do comportamento do classificador.

2. Para reduzir a chance de que o resultado tenha ocorrido por acaso, o experimento foi executado com controle de aleatoriedade por meio de random_state e comparação entre diferentes seeds. Isso permite verificar estabilidade dos resultados e repetir exatamente o mesmo procedimento.

3. Dois possíveis problemas metodológicos são: avaliar com apenas uma partição treino/teste, o que pode introduzir viés na estimativa de desempenho, e testar poucos hiperparâmetros, o que pode limitar a qualidade da comparação entre modelos. Outro ponto é que usar apenas acurácia pode simplificar demais a análise.

4. O pipeline implementado é razoavelmente confiável porque organiza o processo em etapas claras, usa funções separadas, controla a aleatoriedade e mede múltiplas métricas. Ainda assim, ele pode ser melhorado com validação cruzada, busca mais sistemática de hiperparâmetros e análises adicionais por classe.